# Chapter 5 — Files and Exception Handling

**Module**: Programmation Python — ELNI 5.5  
**Weeks**: 11–13  

---

This final chapter closes the loop: you will read real data from files, process it with functions, and write results back. You will also learn to handle errors gracefully so programs remain useful even when input data is imperfect.


---

## 5.1 Working with the Current Directory

The `os` module provides functions for filesystem interaction. The **current working directory** (CWD) is the reference point for all relative file paths.


In [ ]:
import os

# Get and display the current working directory
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

# List directory contents
contents = os.listdir('.')
print(f"\nDirectory contents ({len(contents)} items):")
for item in sorted(contents):
    print(f"  {item}")

In [ ]:
# os.path functions — work with file paths safely

# Build a cross-platform path
# os.path.join() uses the correct separator for the OS (/ on Linux/macOS, \ on Windows)
assets_dir = os.path.join('..', '..', 'assets')
csv_path = os.path.join(assets_dir, 'sensor_data.csv')
log_path = os.path.join(assets_dir, 'sample_log.txt')

print(f"Assets dir : {assets_dir}")
print(f"CSV path   : {csv_path}")

# Check existence before opening
print(f"CSV exists : {os.path.exists(csv_path)}")
print(f"Log exists : {os.path.exists(log_path)}")

# Check if a path is a file or directory
print(f"Is file    : {os.path.isfile(csv_path)}")
print(f"Is dir     : {os.path.isdir(assets_dir)}")

In [ ]:
# Create a directory if it does not exist
output_dir = os.path.join('..', '..', 'output')

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created: {output_dir}")
else:
    print(f"Already exists: {output_dir}")

---

## 5.2 File Manipulation — Reading Files

Files are opened with the built-in `open()` function. The **context manager** (`with` statement) ensures the file is closed automatically, even if an exception occurs.

```python
with open(path, mode, encoding='utf-8') as f:
    # use f here
# file is automatically closed here
```

| Mode | Meaning |
|---|---|
| `'r'` | Read (default) |
| `'w'` | Write (overwrites!) |
| `'a'` | Append |
| `'x'` | Exclusive create (fails if file exists) |


In [ ]:
import os

log_path = os.path.join('..', '..', 'assets', 'sample_log.txt')

# Method 1: .read() — reads entire file as one string
with open(log_path, 'r', encoding='utf-8') as f:
    content = f.read()

print("=== File contents ===")
print(content)
print(f"Total characters: {len(content)}")

In [ ]:
# Method 2: .readlines() — reads all lines into a list
with open(log_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"Number of lines: {len(lines)}")
print(f"First line: {lines[0].strip()!r}")
print(f"Last line : {lines[-1].strip()!r}")

In [ ]:
# Method 3: iterate over the file object — most memory-efficient for large files
error_lines = []

with open(log_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()  # remove newline and whitespace
        if 'ERROR' in line or 'WARNING' in line:
            error_lines.append(line)

print(f"Found {len(error_lines)} error/warning lines:")
for el in error_lines:
    print(f"  {el}")

---

## 5.3 Writing to Files


In [ ]:
import os

output_path = os.path.join('..', '..', 'output', 'report.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Write mode 'w' — creates or overwrites
lines_to_write = [
    "Sensor Report\n",
    "==============\n",
    "Sensor S1: 22.3 °C\n",
    "Sensor S2: 23.1 °C\n",
]

with open(output_path, 'w', encoding='utf-8') as f:
    f.writelines(lines_to_write)

print(f"Written to: {output_path}")

# Append mode 'a' — adds to existing file
with open(output_path, 'a', encoding='utf-8') as f:
    f.write("Sensor S3: 22.8 °C\n")
    f.write("Report complete.\n")

# Verify by reading back
with open(output_path, 'r', encoding='utf-8') as f:
    print(f.read())

---

## 5.4 Copying Files

The `shutil` module (shell utilities) provides high-level file operations. `shutil.copy()` copies file content; `shutil.copy2()` also preserves metadata (timestamps).


In [ ]:
import shutil
import os

source = os.path.join('..', '..', 'assets', 'sensor_data.csv')
destination = os.path.join('..', '..', 'output', 'sensor_data_backup.csv')

os.makedirs(os.path.dirname(destination), exist_ok=True)

# Copy the file
shutil.copy(source, destination)
print(f"Copied {source}")
print(f"     → {destination}")
print(f"Backup exists: {os.path.exists(destination)}")

---

## 5.5 Writing Variables to a File

All file I/O in Python works with strings. To write numeric values, convert them first with `str()` or format them with f-strings.


In [ ]:
import os

# Data from a calculation
beam_id = "B-101"
span_m = 6.5
max_load_kn = 125.0
deflection_mm = 3.47
status = "OK"

output_path = os.path.join('..', '..', 'output', 'beam_results.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    f.write("Beam Design Results\n")
    f.write("=" * 30 + "\n")
    f.write(f"Beam ID        : {beam_id}\n")
    f.write(f"Span           : {span_m:.2f} m\n")
    f.write(f"Max load       : {max_load_kn:.1f} kN\n")
    f.write(f"Max deflection : {deflection_mm:.2f} mm\n")
    f.write(f"Status         : {status}\n")

print(f"Results written to {output_path}")

# Read back and display
with open(output_path, 'r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# Writing a list to CSV format
import csv
import os

records = [
    {"sensor_id": "S1", "temperature_c": 22.3, "status": "OK"},
    {"sensor_id": "S2", "temperature_c": 23.1, "status": "OK"},
    {"sensor_id": "S3", "temperature_c": 24.0, "status": "OK"},
]

csv_output = os.path.join('..', '..', 'output', 'processed.csv')

with open(csv_output, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ["sensor_id", "temperature_c", "status"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)

print(f"CSV written to: {csv_output}")

# Read back with csv module
with open(csv_output, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"  {row['sensor_id']}: {row['temperature_c']} °C — {row['status']}")

---

## 5.6 Reading the Sensor CSV File

Now we read `assets/sensor_data.csv` properly using the `csv` module.


In [ ]:
import csv
import os

csv_path = os.path.join('..', '..', 'assets', 'sensor_data.csv')

data = []
with open(csv_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        data.append({
            "timestamp": row["timestamp"],
            "sensor_id": row["sensor_id"],
            "temperature_c": float(row["temperature_C"]),
            "pressure_kpa": float(row["pressure_kPa"]),
            "status": row["status"]
        })

print(f"Loaded {len(data)} records from CSV.")
print(f"\nFirst record: {data[0]}")
print(f"Last record : {data[-1]}")

---

## 5.7 Exception Handling

Exceptions are Python's mechanism for signalling and handling runtime errors. The structure is:

```python
try:
    # code that might raise an exception
except SomeError as e:
    # handle the error
else:
    # runs only if no exception was raised
finally:
    # always runs (cleanup)
```

Common exception types:

| Exception | When raised |
|---|---|
| `FileNotFoundError` | `open('missing.txt')` |
| `ValueError` | `int('hello')` or invalid argument |
| `ZeroDivisionError` | `10 / 0` |
| `TypeError` | wrong type in operation |
| `KeyError` | accessing a missing dictionary key |
| `IndexError` | accessing a list index out of bounds |


In [ ]:
# Basic try/except
def safe_divide(a, b):
    """Divide a by b, returning None if b is zero."""
    try:
        result = a / b
    except ZeroDivisionError:
        print(f"  Error: cannot divide {a} by zero")
        return None
    return result


print(safe_divide(10, 2))   # 5.0
print(safe_divide(10, 0))   # prints error, returns None

In [ ]:
# Catching multiple exception types
def parse_temperature(raw):
    """
    Parse a temperature string to float.

    Parameters
    ----------
    raw : str
        Raw string from sensor data.

    Returns
    -------
    float or None
    """
    try:
        value = float(raw)
    except ValueError:
        print(f"  ValueError: cannot convert {raw!r} to float")
        return None
    except TypeError:
        print(f"  TypeError: expected a string, got {type(raw).__name__}")
        return None
    else:
        # Runs only if no exception was raised
        if value < -100:
            print(f"  Warning: suspicious value {value}")
        return value


test_inputs = ["22.5", "abc", None, "-999.0", "100"]
for raw in test_inputs:
    result = parse_temperature(raw)
    print(f"  parse_temperature({raw!r}) → {result}")

In [ ]:
# finally — always runs (cleanup code)
def process_file_safely(filepath):
    """
    Read and count lines in a file, handling errors gracefully.

    The finally block demonstrates guaranteed cleanup.
    """
    f = None
    try:
        f = open(filepath, 'r', encoding='utf-8')
        lines = f.readlines()
        print(f"  Successfully read {len(lines)} lines from {filepath}")
        return lines
    except FileNotFoundError:
        print(f"  File not found: {filepath}")
        return []
    except PermissionError:
        print(f"  Permission denied: {filepath}")
        return []
    finally:
        if f is not None:
            f.close()
            print(f"  File handle closed.")
        # This always executes, whether or not an exception occurred.


# Note: in practice, 'with open() as f:' is preferred over manual close()
log_path = os.path.join('..', '..', 'assets', 'sample_log.txt')
lines = process_file_safely(log_path)
print()
lines = process_file_safely("nonexistent_file.txt")

In [ ]:
# Raising exceptions with raise
def compute_stress(force_n, area_m2):
    """
    Compute normal stress with input validation.

    Raises
    ------
    ValueError
        If area_m2 is zero or negative.
    """
    if area_m2 <= 0:
        raise ValueError(f"area_m2 must be positive, but got {area_m2}")
    return force_n / area_m2


# Normal use
try:
    sigma = compute_stress(50000, 0.002)
    print(f"Stress = {sigma/1e6:.2f} MPa")
except ValueError as e:
    print(f"Error: {e}")

# Invalid use
try:
    sigma = compute_stress(50000, -0.002)
except ValueError as e:
    print(f"Caught ValueError: {e}")

In [ ]:
# Full pipeline: read CSV, parse with error handling, compute statistics
import csv
import os


def load_sensor_csv(filepath):
    """
    Load sensor data from a CSV file with robust error handling.

    Returns a list of dicts for valid rows.
    Logs errors for malformed rows instead of crashing.
    """
    records = []
    error_count = 0

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for line_num, row in enumerate(reader, start=2):  # start=2: header is line 1
                try:
                    records.append({
                        "timestamp": row["timestamp"],
                        "sensor_id": row["sensor_id"],
                        "temperature_c": float(row["temperature_C"]),
                        "pressure_kpa": float(row["pressure_kPa"]),
                        "status": row["status"]
                    })
                except (ValueError, KeyError) as e:
                    print(f"  Skipping line {line_num}: {e}")
                    error_count += 1

    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return []

    print(f"Loaded {len(records)} records ({error_count} errors) from {filepath}")
    return records


csv_path = os.path.join('..', '..', 'assets', 'sensor_data.csv')
data = load_sensor_csv(csv_path)

# Compute statistics with error handling
valid_temps = [r["temperature_c"] for r in data if r["status"] == "OK" and r["temperature_c"] > -100]

if valid_temps:
    print(f"Valid readings : {len(valid_temps)}")
    print(f"Average temp   : {sum(valid_temps)/len(valid_temps):.2f} °C")
    print(f"Temperature range: {min(valid_temps):.1f} – {max(valid_temps):.1f} °C")

---

## Chapter 5 Summary

1. **Current directory**: `os.getcwd()`, `os.listdir()`, `os.path.exists()`, `os.path.join()`, `os.makedirs()`.
2. **Reading files**: `open()` in `with` block (modes `'r'`/`'w'`/`'a'`); `.read()`, `.readlines()`, iteration over lines.
3. **Copying files**: `shutil.copy()` and `shutil.copy2()`.
4. **Writing variables**: convert to strings with f-strings; use `csv.DictWriter` for CSV output.
5. **Exception handling**: `try/except/else/finally`; catching specific exceptions; `raise` with a descriptive message; never use bare `except:`.

---

## Course Complete

Congratulations! You have completed all five chapters of Programmation Python — ELNI 5.5. You now have a solid foundation in:

- Python syntax and data types
- Working with lists and dictionaries
- Writing reusable, documented functions
- Reading, writing, and copying files
- Handling errors robustly

These skills are directly applicable to engineering data analysis, automation, and tool development.
